# scverse_misc

## Example 1: a validated Settings class

In [ ]:
# Example 1: Settings (pydantic-settings) validates fields, also on assignment
from typing import Annotated
import pydantic
from pydantic import Field
from scverse_misc import Settings

class MySettings(Settings):
    eps: Annotated[float, Field(gt=0, lt=1)] = 1e-8
    use_optional_feature: bool = False

settings = MySettings()
assert settings.eps == 1e-8 and settings.use_optional_feature is False
try:
    settings.eps = 2.0  # outside (0, 1)
except pydantic.ValidationError as error:
    print("Example 1: eps=2.0 refused:", error.errors()[0]["type"])
else:
    raise AssertionError("an out-of-range setting was accepted")

## Example 2: a deprecated function warns

In [ ]:
# Example 2: deprecated(Deprecation(version, message)) warns with the message on every call
import warnings
from scverse_misc import Deprecation, deprecated

@deprecated(Deprecation("0.2", "Use bar() instead."))
def foo(baz):
    return baz * 2

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    assert foo(21) == 42
assert caught and caught[0].category is FutureWarning and "bar()" in str(caught[0].message), caught
print("Example 2:", caught[0].message)

## Example 3: a deprecated argument warns only when used

In [ ]:
# Example 3: deprecated_arg warns when the old argument is passed, not otherwise
import warnings
from scverse_misc import Deprecation, deprecated_arg

@deprecated_arg("copy", Deprecation("0.3", "copy is ignored."))
def scale(x, factor=2, copy=None):
    return x * factor

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    assert scale(3) == 6
    assert scale(3, copy=True) == 6
assert len(caught) == 1 and "copy" in str(caught[0].message), [str(w.message) for w in caught]
print("Example 3:", caught[0].message)

## Example 4: argument aliases

In [ ]:
# Example 4: arg_alias maps every spelling of a Literal choice to its canonical value
from typing import Literal
from scverse_misc import arg_alias

@arg_alias("axis")
def reduce(x: float, axis: Literal[0, "obs"] | Literal[1, "var", "vars"]):
    return axis

assert reduce(1.0, 0) == reduce(1.0, "obs") == 0
assert reduce(1.0, 1) == reduce(1.0, "var") == reduce(1.0, "vars") == 1
print("Example 4: obs -> 0, var/vars -> 1")

## Example 5: suppress, the exception-silencing context manager

In [ ]:
# Example 5: suppress(...) swallows only the named exception types
from scverse_misc import suppress

with suppress(KeyError):
    {}["missing"]
try:
    with suppress(KeyError):
        raise ValueError("not suppressed")
except ValueError as error:
    print("Example 5: KeyError suppressed; ValueError passed through:", error)
else:
    raise AssertionError("suppress swallowed an exception it was not given")